# Day 13 — Conditional logic: when/then/otherwise

*100 Days of Polars · Phase 2: Expressions*

- **Post:** [Polars Conditional Logic With when/then/otherwise](https://python.plainenglish.io/day-13-polars-conditional-logic-with-when-then-otherwise-b00755d90ad1)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [65]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


`pl.when(cond).then(value).otherwise(default)` is SQL's `CASE WHEN` and pandas' `np.where` / `np.select`, as one expression.
Conditions are checked **top to bottom**; the first match wins.

In [66]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
orders = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
payments = pl.read_csv("data/raw/olist_order_payments_dataset.csv")

## 1. One condition

In [67]:
items.select(
    "price",
    is_expensive=pl.when(pl.col("price") > 500).then(pl.lit("yes")).otherwise(pl.lit("no")),
).head(5)

price,is_expensive
f64,str
58.9,"""no"""
239.9,"""no"""
199.0,"""no"""
12.99,"""no"""
199.9,"""no"""


## 2. A chain of conditions: price bands

In [68]:
price_band = (
    pl.when(pl.col("price") < 50).then(pl.lit("low"))
    .when(pl.col("price") < 200).then(pl.lit("mid"))
    .otherwise(pl.lit("high"))
)

items.with_columns(band=price_band).group_by("band").agg(n=pl.len(), avg_price=pl.col("price").mean().round(2)).sort("avg_price")

band,n,avg_price
str,u32,f64
"""low""",39024,31.3
"""mid""",60153,105.48
"""high""",13473,447.19


The second `when` only sees rows the first one did not match, so `< 200` really means `50 <= price < 200`.

## 3. Gotcha: strings in `then` are column names

In [69]:
try:
    items.select(pl.when(pl.col("price") > 500).then("yes").otherwise("no"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

ColumnNotFoundError -> unable to find column "yes"; valid columns: ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"]


Wrap string values in `pl.lit(...)`. Numbers, booleans and dates are fine as plain Python values.

In [70]:
# No otherwise -> null for unmatched rows
items.select("price", flag=pl.when(pl.col("price") > 500).then(True)).head(5)

price,flag
f64,bool
58.9,null
239.9,null
199.0,null
12.99,null
199.9,null


## 4. Real logic: delivery status

In [71]:
delivery_status = (
    pl.when(pl.col("order_delivered_customer_date").is_null()).then(pl.lit("not delivered"))
    .when(pl.col("order_delivered_customer_date") > pl.col("order_estimated_delivery_date")).then(pl.lit("late"))
    .otherwise(pl.lit("on time"))
)

orders.group_by(status=delivery_status).len().sort("len", descending=True)

status,len
str,u32
"""on time""",88649
"""late""",7827
"""not delivered""",2965


In [72]:
# Several predicates in one when() are ANDed
orders.select(
    stuck_in_transit=pl.when(
        pl.col("order_status") == "shipped",
        pl.col("order_delivered_customer_date").is_null(),
    ).then(1).otherwise(0).sum()
)

stuck_in_transit
i32
1107


## 5. Conditional values, not just labels

In [73]:
# Cash actually paid: vouchers are not cash
payments.select(
    total_paid=pl.col("payment_value").sum().round(2),
    cash_paid=pl.when(pl.col("payment_type") == "voucher").then(0).otherwise(pl.col("payment_value")).sum().round(2),
)

total_paid,cash_paid
f64,f64
1.6009e7,1.5629e7


## 6. When a lookup is better: `replace_strict`

If every branch is "this value maps to that value", use a mapping instead of a long chain.

In [74]:
labels = {"credit_card": "Card", "debit_card": "Card", "boleto": "Bank slip", "voucher": "Voucher"}

payments.group_by(
    method=pl.col("payment_type").replace_strict(labels, default="Other")
).len().sort("len", descending=True)

method,len
str,u32
"""Card""",78324
"""Bank slip""",19784
"""Voucher""",5775
"""Other""",3


## 7. Where the output name comes from

The output name comes from the **first `then`**, never from the condition. The same leftmost rule as before, applied to `when/then`.

In [75]:
e1 = pl.when(pl.col("price") > 500).then(pl.col("price") * 0.9).otherwise(pl.col("price"))
e2 = pl.when(pl.col("price") > 500).then(pl.lit("high")).otherwise(pl.lit("low"))
e3 = pl.when(pl.col("freight_value") > 20).then(pl.col("price"))

print(e1.meta.output_name())  # price
print(e2.meta.output_name())  # literal: labels almost always need a name
print(e3.meta.output_name())  # price, not freight_value: the condition never names the output

price
literal
price


In [76]:
items.select(discounted=e1, band=e2).head(3)

discounted,band
f64,str
58.9,"""low"""
239.9,"""low"""
199.0,"""low"""


In [77]:
# Branches with different types get a common supertype, silently: here String
mixed = (
    pl.when(pl.col("price") > 500).then(pl.lit("high"))
    .when(pl.col("price") > 200).then(pl.col("price") * 0.5)
    .otherwise(pl.lit("low"))
)
items.select("price", band=mixed).head(3)   # 239.9 -> "119.95", as text

price,band
f64,str
58.9,"""low"""
239.9,"""119.95"""
199.0,"""low"""


## 8. Many columns at once, and `name.keep()`

A `then` or `otherwise` can hold several columns (a selector or `pl.all()`). Then every expanded output takes its name from the first `then`.

Example: products with no weight have unreliable dimensions, so blank out the three `_cm` columns for them.

In [78]:
import polars.selectors as cs

products = pl.read_csv("data/raw/olist_products_dataset.csv")
no_weight = pl.col("product_weight_g").is_null() | (pl.col("product_weight_g") == 0)
print(products.filter(no_weight).height, "products with no weight")

6 products with no weight


In [79]:
# Column expression in the FIRST then: each output keeps its own column name
keep_dims = pl.when(~no_weight).then(cs.ends_with("_cm"))   # no otherwise -> null
print(keep_dims.meta.has_multiple_outputs())
products.with_columns(keep_dims).filter(no_weight).select("product_weight_g", cs.ends_with("_cm")).head()

True


product_weight_g,product_length_cm,product_height_cm,product_width_cm
i64,i64,i64,i64
null,null,null,null
0,null,null,null
0,null,null,null
0,null,null,null
null,null,null,null


In [80]:
# Same logic reversed: a constant in the first then -> every output is called "literal"
blank_dims = pl.when(no_weight).then(None).otherwise(cs.ends_with("_cm"))
try:
    products.with_columns(blank_dims)
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

ComputeError -> the name 'literal' passed to `LazyFrame.with_columns` is duplicate


In [81]:
# CHECK: same values, only the names differ
print("with name.keep():", products.select(blank_dims.name.keep()).columns)   # original names -> replace in with_columns

# name.suffix does NOT go back to the column name: it suffixes the current name "literal"
try:
    products.select(blank_dims.name.suffix("_clean"))
except Exception as err:
    print("with name.suffix():", type(err).__name__, "->", str(err).splitlines()[0])

with name.keep(): ['product_length_cm', 'product_height_cm', 'product_width_cm']
with name.suffix(): DuplicateError -> projections contained duplicate output name 'literal_clean'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


In [82]:
# Two ways to get NEW suffixed columns (originals kept)
# a) keep first, then suffix  b) flip the logic so the column expression is the first then
a = products.select(blank_dims.name.keep().name.suffix("_clean")).columns
b = products.select(pl.when(~no_weight).then(cs.ends_with("_cm")).name.suffix("_clean")).columns
print("a)", a)
print("b)", b)

a) ['product_length_cm_clean', 'product_height_cm_clean', 'product_width_cm_clean']
b) ['product_length_cm_clean', 'product_height_cm_clean', 'product_width_cm_clean']


In [83]:
# with_columns + name.keep() overwrites the three _cm columns in the result (products itself is unchanged)
cleaned = products.with_columns(blank_dims.name.keep())
print(cleaned.filter(no_weight).select(cs.ends_with("_cm")).null_count())
print(products.filter(no_weight).select(cs.ends_with("_cm")).null_count())

shape: (1, 3)
┌───────────────────┬───────────────────┬──────────────────┐
│ product_length_cm ┆ product_height_cm ┆ product_width_cm │
│ ---               ┆ ---               ┆ ---              │
│ u32               ┆ u32               ┆ u32              │
╞═══════════════════╪═══════════════════╪══════════════════╡
│ 6                 ┆ 6                 ┆ 6                │
└───────────────────┴───────────────────┴──────────────────┘
shape: (1, 3)
┌───────────────────┬───────────────────┬──────────────────┐
│ product_length_cm ┆ product_height_cm ┆ product_width_cm │
│ ---               ┆ ---               ┆ ---              │
│ u32               ┆ u32               ┆ u32              │
╞═══════════════════╪═══════════════════╪══════════════════╡
│ 2                 ┆ 2                 ┆ 2                │
└───────────────────┴───────────────────┴──────────────────┘


## 9. `otherwise` is optional, and there is only one

Without `otherwise`, unmatched rows get null (the docs: "a default of `.otherwise(None)` is used"). `.otherwise()` returns a plain expression, so nothing can be chained after it. For more cases, add more `.when().then()` pairs **before** the single `otherwise`.

In [84]:
flag = pl.when(pl.col("price") > 500).then(True)   # no otherwise
items.select(n_null=flag.is_null().sum(), n_true=flag.sum())

n_null,n_true
u32,u32
109434,3216


In [85]:
try:
    pl.when(pl.col("price") > 500).then(1).otherwise(2).otherwise(3)
except AttributeError as err:
    print("AttributeError ->", err)

AttributeError -> 'Expr' object has no attribute 'otherwise'


In [86]:
# More cases = more when/then pairs. A when nested inside otherwise does the same, less readably.
flat = (
    pl.when(pl.col("price") < 50).then(pl.lit("low"))
    .when(pl.col("price") < 200).then(pl.lit("mid"))
    .otherwise(pl.lit("high"))
)
nested = (
    pl.when(pl.col("price") < 50).then(pl.lit("low"))
    .otherwise(pl.when(pl.col("price") < 200).then(pl.lit("mid")).otherwise(pl.lit("high")))
)
print(items.select(flat.alias("band")).equals(items.select(nested.alias("band"))))

True


## 10. Keyword conditions

`pl.when(col_name=value)` is shorthand for an equality check. Several conditions (positional or keyword) are ANDed.

In [87]:
orders.select(
    delivered_kw=pl.when(order_status="delivered").then(1).otherwise(0).sum(),
    delivered_eq=pl.when(pl.col("order_status") == "delivered").then(1).otherwise(0).sum(),
)

delivered_kw,delivered_eq
i32,i32
96478,96478


## 11. Gotcha: a `when` is not a guard

The `pl.when` docstring warns: "Polars computes all expressions passed to `when-then-otherwise` in parallel and filters afterwards. This means each expression must be valid on its own, regardless of the conditions in the `when-then-otherwise` chain."

That is a contract, not a prediction: Polars is *allowed* to run every branch on every row. What actually raises depends on the operation and the version.

In [88]:
raw = pl.DataFrame({"qty": ["10", "20", "3x"]})
is_number = pl.col("qty").str.contains(r"^\d+$")

# Control: a strict cast with no when fails on "3x"
try:
    raw.select(pl.col("qty").cast(pl.Int64))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

InvalidOperationError -> conversion from `str` to `i64` failed in column 'qty' for 1 out of 3 values: ["3x"]


In [89]:
# Inside a when, the same strict cast did NOT fail in Polars 2.0.0: "3x" just becomes null.
# An engine detail, not a promise: the docs still say every branch must be valid on its own.
raw.with_columns(qty_int=pl.when(is_number).then(pl.col("qty").cast(pl.Int64)))

qty,qty_int
str,i64
"""10""",10
"""20""",20
"""3x""",null


In [90]:
# Is it the optimizer? Compare the plans: they are identical
q = raw.lazy().with_columns(qty_int=pl.when(is_number).then(pl.col("qty").cast(pl.Int64)))
print("=== OPTIMIZED ===");   print(q.explain())
print("=== UNOPTIMIZED ==="); print(q.explain(optimized=False))

=== OPTIMIZED ===
 WITH_COLUMNS:
 [when(col("qty").str.contains(["^\d+$"])).then(col("qty").strict_cast(Int64)).otherwise(null.cast(Int64)).alias("qty_int")] 
  DF ["qty"]; PROJECT */1 COLUMNS
=== UNOPTIMIZED ===
 WITH_COLUMNS:
 [when(col("qty").str.contains(["^\d+$"])).then(col("qty").strict_cast(Int64)).otherwise(null.cast(Int64)).alias("qty_int")] 
  DF ["qty"]; PROJECT */1 COLUMNS


In [91]:
# No optimizations, and both engines: same result, no error -> it happens at evaluation time
print(q.collect(optimizations=pl.QueryOptFlags.none()))
print(q.collect(engine="in-memory"))
print(q.collect(engine="streaming"))

shape: (3, 2)
┌─────┬─────────┐
│ qty ┆ qty_int │
│ --- ┆ ---     │
│ str ┆ i64     │
╞═════╪═════════╡
│ 10  ┆ 10      │
│ 20  ┆ 20      │
│ 3x  ┆ null    │
└─────┴─────────┘
shape: (3, 2)
┌─────┬─────────┐
│ qty ┆ qty_int │
│ --- ┆ ---     │
│ str ┆ i64     │
╞═════╪═════════╡
│ 10  ┆ 10      │
│ 20  ┆ 20      │
│ 3x  ┆ null    │
└─────┴─────────┘
shape: (3, 2)
┌─────┬─────────┐
│ qty ┆ qty_int │
│ --- ┆ ---     │
│ str ┆ i64     │
╞═════╪═════════╡
│ 10  ┆ 10      │
│ 20  ┆ 20      │
│ 3x  ┆ null    │
└─────┴─────────┘


In [92]:
# Does another strict function behave the same? Date parsing inside a when:
dates = pl.DataFrame({"d": ["2017-11-20", "not a date"]})
looks_like_date = pl.col("d").str.contains(r"^\d{4}-\d{2}-\d{2}$")
try:
    print(dates.with_columns(parsed=pl.when(looks_like_date).then(pl.col("d").str.to_date("%Y-%m-%d"))))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

shape: (2, 2)
┌────────────┬────────────┐
│ d          ┆ parsed     │
│ ---        ┆ ---        │
│ str        ┆ date       │
╞════════════╪════════════╡
│ 2017-11-20 ┆ 2017-11-20 │
│ not a date ┆ null       │
└────────────┴────────────┘


In [93]:
# Here cast(strict=False) alone would do; the when adds nothing.
# A when earns its place when the branches compute DIFFERENT things.
# Two formats: plain numbers ("10") and numbers with a unit suffix ("3x").
# Both branches run on every row, so both must be non-strict.
raw.with_columns(
    qty_int=pl.when(is_number)
    .then(pl.col("qty").cast(pl.Int64, strict=False))                                 # "10" -> 10 (also sees "3x")
    .otherwise(pl.col("qty").str.strip_chars_end("x").cast(pl.Int64, strict=False))   # "3x" -> 3  (also sees "10")
)

qty,qty_int
str,i64
"""10""",10
"""20""",20
"""3x""",3


## Try it yourself

1. Bucket orders by delivery days: `<7`, `7-14`, `15-30`, `>30`, `not delivered`.
2. Add `installment_type`: `"single"` for 1 installment, `"short"` for 2-6, `"long"` otherwise.
3. What does `price_band` return for a null price? Test it on `pl.DataFrame({"price": [10.0, None, 300.0]})`.